# Notebook 05 — ML vs DL Comparison & Final Analysis

**Goal:** Merge all results from the ML and DL modeling phases into a unified analysis.
No training happens here — this notebook is pure analysis, visualization, and conclusions.

**Inputs:**
- `results/ml_baselines.csv` (from nb03a)
- `results/ml_imbalance.csv` (from nb03b)
- `results/ml_tuned.csv` (from nb03c)
- `results/ml_ensemble.csv` (from nb03d)
- `results/ml_final_test.csv` (from nb03d)
- `results/dl_architectures.csv` (from nb04a)
- `results/dl_tuned.csv` (from nb04b)
- `results/dl_final_test.csv` (from nb04b)
- `models/` (saved models for SHAP analysis)

**Outputs:** Final comparison charts, tables, and conclusions for the PFA report.


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, os, joblib
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

os.makedirs('results/final', exist_ok=True)
print("Setup OK")


## 1. Load All Results

In [ ]:
# ── Load all scoreboard CSVs ──
ml_baselines = pd.read_csv('results/ml_baselines.csv')
ml_imbalance = pd.read_csv('results/ml_imbalance.csv')
ml_tuned     = pd.read_csv('results/ml_tuned.csv')
ml_ensemble  = pd.read_csv('results/ml_ensemble.csv')
dl_arch      = pd.read_csv('results/dl_architectures.csv')
dl_tuned     = pd.read_csv('results/dl_tuned.csv')

# Tag each with source
ml_baselines['source'] = 'nb03a'
ml_imbalance['source'] = 'nb03b'
ml_tuned['source']     = 'nb03c'
ml_ensemble['source']  = 'nb03d'
dl_arch['source']      = 'nb04a'
dl_tuned['source']     = 'nb04b'

# Combine into master scoreboard
all_experiments = pd.concat([
    ml_baselines, ml_imbalance, ml_tuned, ml_ensemble, dl_arch, dl_tuned
], ignore_index=True)

# Add ML/DL tag
all_experiments['type'] = all_experiments['source'].apply(
    lambda x: 'DL' if x.startswith('nb04') else 'ML'
)

print(f"Total experiments: {len(all_experiments)}")
print(f"  ML: {(all_experiments['type']=='ML').sum()}")
print(f"  DL: {(all_experiments['type']=='DL').sum()}")
print(f"\nSources:")
print(all_experiments['source'].value_counts().to_string())

# ── Load test results ──
ml_test = pd.read_csv('results/ml_final_test.csv')
dl_test = pd.read_csv('results/dl_final_test.csv')
ml_test['type'] = 'ML'
dl_test['type'] = 'DL'
all_test = pd.concat([ml_test, dl_test], ignore_index=True)

print(f"\nTest set results:")
print(all_test.to_string(index=False))


## 2. Master Scoreboard

Every experiment across all notebooks, sorted by macro F1.


In [ ]:
# Full scoreboard sorted by F1
master = all_experiments.sort_values('cv_f1_mean', ascending=False)
master.to_csv('results/final/master_scoreboard.csv', index=False)

print(f"MASTER SCOREBOARD ({len(master)} experiments)")
print("=" * 100)
print(master[['source', 'type', 'model', 'strategy', 'cv_f1_mean', 'cv_f1_std', 'notes']]
      .to_string(index=False))


In [ ]:
# Best per source notebook
print("\nBEST PER NOTEBOOK:")
print("-" * 80)
for src in ['nb03a', 'nb03b', 'nb03c', 'nb03d', 'nb04a', 'nb04b']:
    sub = master[master['source'] == src]
    if len(sub) > 0:
        best = sub.iloc[0]
        print(f"  {src}: {best['model']:22s} | {best['strategy']:10s} | F1={best['cv_f1_mean']:.4f}")


## 3. Impact of Problem Framing

The single most impactful decision across all experiments was how we defined the classification problem.


In [ ]:
# Best F1 per strategy (across all models, ML + DL)
strategy_best = master.groupby('strategy').agg(
    best_f1=('cv_f1_mean', 'max'),
    best_model=('model', lambda x: master.loc[x.index[master.loc[x.index, 'cv_f1_mean'].argmax()], 'model']),
    best_type=('type', lambda x: master.loc[x.index[master.loc[x.index, 'cv_f1_mean'].argmax()], 'type']),
).sort_values('best_f1', ascending=False)

print("BEST F1 PER PROBLEM FRAMING:")
print(strategy_best.to_string())

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

# Group strategies
framing_order = ['binary', '3class', 'under_10k', 'under_15k', 'under_6k',
                 'under_15k+SMOTE', 'SMOTE', 'SMOTE+Tomek', '4class']
framing_labels = {
    'binary': 'Binary\n(success vs failed)',
    '3class': '3-class\n(closed/acquired/ipo)',
    'under_10k': 'Undersample 10k',
    'under_15k': 'Undersample 15k',
    'under_6k': 'Undersample 6k',
    'under_15k+SMOTE': 'Under 15k + SMOTE',
    'SMOTE': 'SMOTE',
    'SMOTE+Tomek': 'SMOTE + Tomek',
    '4class': '4-class\n(original, 80% operating)',
}

best_per_framing = []
for strat in framing_order:
    sub = master[master['strategy'] == strat]
    if len(sub) > 0:
        best = sub.iloc[0]
        best_per_framing.append({
            'strategy': strat,
            'label': framing_labels.get(strat, strat),
            'f1': best['cv_f1_mean'],
            'model': best['model'],
            'type': best['type'],
        })

bpf = pd.DataFrame(best_per_framing).sort_values('f1', ascending=True)

colors_map = {
    'binary': '#1D9E75', '3class': '#534AB7',
    '4class': '#888780', 'SMOTE': '#D85A30', 'SMOTE+Tomek': '#D85A30',
}
colors = [colors_map.get(s, '#378ADD') for s in bpf['strategy']]

bars = ax.barh(bpf['label'], bpf['f1'], color=colors, edgecolor='white', height=0.6)
for bar, row in zip(bars, bpf.itertuples()):
    ax.text(row.f1 + 0.005, bar.get_y() + bar.get_height()/2,
            f'{row.f1:.4f} ({row.model})', va='center', fontsize=10)

ax.set_xlabel('Best Macro F1 Score', fontsize=12)
ax.set_title('Impact of Problem Framing on Best Achievable F1', fontsize=14)
ax.set_xlim(0, 0.85)
ax.axvline(x=0.5, color='gray', linestyle=':', alpha=0.3)

plt.tight_layout()
plt.savefig('results/final/problem_framing_impact.png', bbox_inches='tight')
plt.show()

# Compute the gain
if len(bpf) >= 2:
    worst = bpf.iloc[0]['f1']
    best = bpf.iloc[-1]['f1']
    print(f"\nGain from problem reframing: {worst:.4f} → {best:.4f} (+{best-worst:.4f})")


## 4. ML vs DL — Head-to-Head Comparison

The core question of the PFA: on the same data, same features, same evaluation protocol, which family wins?


In [ ]:
# ── Best ML vs best DL per strategy ──
head2head = []
for strat in ['3class', 'binary']:
    for typ in ['ML', 'DL']:
        sub = master[(master['strategy'] == strat) & (master['type'] == typ)]
        if len(sub) > 0:
            best = sub.iloc[0]
            head2head.append({
                'strategy': strat,
                'type': typ,
                'model': best['model'],
                'f1': best['cv_f1_mean'],
                'std': best['cv_f1_std'],
                'source': best['source'],
            })

h2h = pd.DataFrame(head2head)

print("ML vs DL — Best Model per Strategy (CV scores):")
print("=" * 70)
print(h2h.to_string(index=False))

# Compute gaps
for strat in ['3class', 'binary']:
    ml_f1 = h2h[(h2h['strategy']==strat) & (h2h['type']=='ML')]['f1'].values
    dl_f1 = h2h[(h2h['strategy']==strat) & (h2h['type']=='DL')]['f1'].values
    if len(ml_f1) > 0 and len(dl_f1) > 0:
        gap = ml_f1[0] - dl_f1[0]
        winner = "ML" if gap > 0 else "DL"
        print(f"\n  {strat}: ML={ml_f1[0]:.4f}, DL={dl_f1[0]:.4f}, gap={gap:+.4f} → {winner} wins")


In [ ]:
# ── Head-to-head bar chart ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, strat in enumerate(['3class', 'binary']):
    ax = axes[idx]
    strat_data = h2h[h2h['strategy'] == strat].sort_values('f1', ascending=True)

    colors = ['#378ADD' if t == 'ML' else '#D85A30' for t in strat_data['type']]
    bars = ax.barh(
        strat_data['type'] + '\n' + strat_data['model'],
        strat_data['f1'], color=colors, edgecolor='white', height=0.5
    )
    for bar, row in zip(bars, strat_data.itertuples()):
        ax.text(row.f1 + 0.003, bar.get_y() + bar.get_height()/2,
                f'{row.f1:.4f}', va='center', fontsize=11, fontweight='bold')

    title = '3-class (closed / acquired / ipo)' if strat == '3class' else 'Binary (success vs failed)'
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Macro F1')
    ax.set_xlim(0, 0.85)

plt.suptitle('ML vs DL — Head-to-Head', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/final/ml_vs_dl_headtohead.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Test set comparison ──
print("\n" + "=" * 70)
print("FINAL TEST SET RESULTS (one shot, no re-tuning)")
print("=" * 70)
print(all_test.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
all_test_sorted = all_test.sort_values('test_f1', ascending=True)
colors = ['#378ADD' if t == 'ML' else '#D85A30' for t in all_test_sorted['type']]
bars = ax.barh(
    all_test_sorted['type'] + ' | ' + all_test_sorted['model'] + ' (' + all_test_sorted['strategy'] + ')',
    all_test_sorted['test_f1'], color=colors, edgecolor='white', height=0.5
)
for bar, val in zip(bars, all_test_sorted['test_f1']):
    ax.text(val + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=11, fontweight='bold')
ax.set_xlabel('Test Macro F1', fontsize=12)
ax.set_title('Final Test Set Performance — ML vs DL', fontsize=14)
ax.set_xlim(0, 0.85)
plt.tight_layout()
plt.savefig('results/final/test_set_comparison.png', bbox_inches='tight')
plt.show()


## 5. Model Family Comparison

How does each model family perform across problem framings?


In [ ]:
# ── All models on 3-class and binary ──
model_families = master[master['strategy'].isin(['3class', 'binary'])].copy()

# Keep only the best score per model per strategy
best_per_model = model_families.groupby(['model', 'strategy', 'type']).agg(
    f1=('cv_f1_mean', 'max')
).reset_index()

# Pivot for heatmap
pivot_3c = best_per_model[best_per_model['strategy']=='3class'].set_index('model')['f1'].sort_values(ascending=False)
pivot_bin = best_per_model[best_per_model['strategy']=='binary'].set_index('model')['f1'].sort_values(ascending=False)

# Combine
all_models = set(pivot_3c.index) | set(pivot_bin.index)
comparison_df = pd.DataFrame({
    '3-class': pivot_3c,
    'binary': pivot_bin,
}).reindex(all_models)

# Sort by binary (or 3-class if binary is missing)
comparison_df['sort_key'] = comparison_df['binary'].fillna(comparison_df['3-class'])
comparison_df = comparison_df.sort_values('sort_key', ascending=False).drop(columns='sort_key')

# Add type tag
type_map = best_per_model.drop_duplicates('model').set_index('model')['type'].to_dict()
comparison_df['type'] = comparison_df.index.map(type_map)

print("ALL MODELS — BEST F1 PER STRATEGY:")
print("=" * 60)
print(comparison_df.to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(8, max(6, len(comparison_df) * 0.4)))
# Add type to index for display
display_df = comparison_df[['3-class', 'binary']].copy()
display_df.index = [f"{'🌲' if type_map.get(m,'ML')=='ML' else '🧠'} {m}" for m in display_df.index]

sns.heatmap(display_df, annot=True, fmt='.4f', cmap='YlGn', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Macro F1'},
            mask=display_df.isna())
ax.set_title('All Models — Macro F1 by Strategy\n(🌲 = ML, 🧠 = DL)', fontsize=13)
ax.set_ylabel(''); ax.set_xlabel('')
plt.tight_layout()
plt.savefig('results/final/all_models_heatmap.png', bbox_inches='tight')
plt.show()


## 6. Experiment Progression

How did performance improve across the modeling phases?


In [ ]:
# Best F1 at each phase
phase_order = ['nb03a', 'nb03b', 'nb03c', 'nb03d', 'nb04a', 'nb04b']
phase_labels = {
    'nb03a': 'ML Baselines',
    'nb03b': 'Imbalance\nStrategies',
    'nb03c': 'ML Tuning',
    'nb03d': 'ML Ensemble',
    'nb04a': 'DL Architectures',
    'nb04b': 'DL Tuning',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for idx, strat in enumerate(['3class', 'binary']):
    ax = axes[idx]

    ml_phases = []
    dl_phases = []
    for src in phase_order:
        sub = master[(master['source'] == src) & (master['strategy'] == strat)]
        if len(sub) > 0:
            best_f1 = sub['cv_f1_mean'].max()
            if src.startswith('nb03'):
                ml_phases.append((phase_labels[src], best_f1))
            else:
                dl_phases.append((phase_labels[src], best_f1))

    if ml_phases:
        labels, vals = zip(*ml_phases)
        ax.plot(labels, vals, 'o-', color='#378ADD', linewidth=2, markersize=8, label='ML')
        for i, v in enumerate(vals):
            ax.annotate(f'{v:.4f}', (i, v), textcoords='offset points',
                       xytext=(0, 12), ha='center', fontsize=9, color='#378ADD')

    if dl_phases:
        # Offset x positions for DL
        x_offset = len(ml_phases)
        labels_dl, vals_dl = zip(*dl_phases)
        x_pos = list(range(x_offset, x_offset + len(dl_phases)))
        ax.plot(x_pos, vals_dl, 's-', color='#D85A30', linewidth=2, markersize=8, label='DL')
        for i, v in zip(x_pos, vals_dl):
            ax.annotate(f'{v:.4f}', (i, v), textcoords='offset points',
                       xytext=(0, 12), ha='center', fontsize=9, color='#D85A30')

        # Combined x labels
        all_labels = list(labels) + list(labels_dl)
        ax.set_xticks(range(len(all_labels)))
        ax.set_xticklabels(all_labels, fontsize=9)

    title = '3-class' if strat == '3class' else 'Binary'
    ax.set_title(f'{title} — Performance Progression', fontsize=12)
    ax.set_ylabel('Best Macro F1')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('How Performance Improved Across Phases', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/final/progression.png', bbox_inches='tight')
plt.show()


## 7. Feature Importance — ML (SHAP) vs DL (TabNet Attention)

Comparing what each model family considers important.
If they agree, it validates the features. If they disagree, it reveals different learning mechanisms.


In [ ]:
# ── ML: XGBoost feature importance ──
try:
    xgb_model = joblib.load('models/xgb_tuned_3class.pkl')
    feature_names = joblib.load('artifacts/feature_names.pkl')

    # XGBoost built-in importance (gain-based)
    xgb_imp = xgb_model.feature_importances_
    xgb_imp_df = pd.DataFrame({
        'feature': feature_names,
        'xgb_importance': xgb_imp / xgb_imp.sum(),  # normalize to sum=1
    }).sort_values('xgb_importance', ascending=False)

    print("XGBoost Feature Importance (top 10):")
    print(xgb_imp_df.head(10).to_string(index=False))
    ml_imp_available = True
except FileNotFoundError:
    print("XGBoost model not found — run nb03d first.")
    ml_imp_available = False


In [ ]:
# ── DL: TabNet feature importance ──
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    tabnet_model = TabNetClassifier()
    tabnet_model.load_model('models/tabnet_tuned_3class.zip')
    feature_names = joblib.load('artifacts/feature_names.pkl')

    tabnet_imp = tabnet_model.feature_importances_
    tabnet_imp_df = pd.DataFrame({
        'feature': feature_names,
        'tabnet_importance': tabnet_imp / tabnet_imp.sum(),
    }).sort_values('tabnet_importance', ascending=False)

    print("TabNet Feature Importance (top 10):")
    print(tabnet_imp_df.head(10).to_string(index=False))
    dl_imp_available = True
except Exception as e:
    print(f"TabNet model not loaded: {e}")
    print("Run nb04b first, or the model was saved with a different path.")
    dl_imp_available = False


In [ ]:
# ── Side-by-side comparison ──
if ml_imp_available and dl_imp_available:
    merged_imp = xgb_imp_df.merge(tabnet_imp_df, on='feature')
    merged_imp = merged_imp.sort_values('xgb_importance', ascending=False)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # XGBoost importance
    ax = axes[0]
    top_xgb = merged_imp.nlargest(12, 'xgb_importance')
    ax.barh(top_xgb['feature'][::-1], top_xgb['xgb_importance'][::-1],
            color='#378ADD', edgecolor='white')
    ax.set_title('XGBoost (Gain-based)', fontsize=12)
    ax.set_xlabel('Normalized Importance')

    # TabNet importance
    ax = axes[1]
    top_tab = merged_imp.nlargest(12, 'tabnet_importance')
    ax.barh(top_tab['feature'][::-1], top_tab['tabnet_importance'][::-1],
            color='#D85A30', edgecolor='white')
    ax.set_title('TabNet (Attention-based)', fontsize=12)
    ax.set_xlabel('Normalized Importance')

    # Scatter: correlation between the two
    ax = axes[2]
    ax.scatter(merged_imp['xgb_importance'], merged_imp['tabnet_importance'],
               s=60, alpha=0.7, color='#534AB7')
    for _, row in merged_imp.nlargest(5, 'xgb_importance').iterrows():
        ax.annotate(row['feature'], (row['xgb_importance'], row['tabnet_importance']),
                   fontsize=8, ha='left')
    corr = merged_imp['xgb_importance'].corr(merged_imp['tabnet_importance'])
    ax.set_xlabel('XGBoost Importance')
    ax.set_ylabel('TabNet Importance')
    ax.set_title(f'Correlation: r = {corr:.3f}', fontsize=12)

    plt.suptitle('Feature Importance — ML vs DL', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('results/final/feature_importance_comparison.png', bbox_inches='tight')
    plt.show()

    print(f"\nCorrelation between XGBoost and TabNet importance: r = {corr:.3f}")
    if corr > 0.7:
        print("→ Strong agreement: both models rely on similar features.")
    elif corr > 0.4:
        print("→ Moderate agreement: some overlap but also differences in what each model values.")
    else:
        print("→ Weak agreement: the models capture fundamentally different patterns.")

    # Save
    merged_imp.to_csv('results/final/feature_importance_comparison.csv', index=False)
elif ml_imp_available:
    print("Only ML importance available — skipping comparison plot.")
    fig, ax = plt.subplots(figsize=(10, 6))
    top_xgb = xgb_imp_df.nlargest(15, 'xgb_importance')
    ax.barh(top_xgb['feature'][::-1], top_xgb['xgb_importance'][::-1],
            color='#378ADD', edgecolor='white')
    ax.set_title('XGBoost Feature Importance (Gain-based)', fontsize=12)
    ax.set_xlabel('Normalized Importance')
    plt.tight_layout()
    plt.show()
else:
    print("No model importance available — run nb03d and nb04b first.")


## 8. Feature Ablation Recap

From nb03c: which feature groups carry signal?


In [ ]:
# Load ablation results from tuned scoreboard
try:
    tuned = pd.read_csv('results/ml_tuned.csv')
    ablation = tuned[tuned['model'].str.startswith('drop_') | (tuned['model'] == 'XGB-tuned-ALL')]

    if len(ablation) > 0:
        # Parse ablation results
        baseline_f1 = ablation[ablation['model'].str.contains('ALL')]['cv_f1_mean'].values
        if len(baseline_f1) == 0:
            baseline_f1 = ablation['cv_f1_mean'].max()
        else:
            baseline_f1 = baseline_f1[0]

        drops = ablation[ablation['model'].str.startswith('drop_')].copy()
        drops['group'] = drops['model'].str.replace('drop_', '')
        drops['delta'] = drops['cv_f1_mean'] - baseline_f1

        drops = drops.sort_values('delta')

        fig, ax = plt.subplots(figsize=(10, 5))
        colors = ['#1D9E75' if d >= 0 else '#E24B4A' for d in drops['delta']]
        ax.barh(drops['group'], drops['delta'], color=colors, edgecolor='white', height=0.5)
        ax.axvline(x=0, color='black', linewidth=0.5)
        for i, (g, d) in enumerate(zip(drops['group'], drops['delta'])):
            ax.text(d + (0.001 if d >= 0 else -0.001), i,
                    f'{d:+.4f}', va='center', ha='left' if d >= 0 else 'right', fontsize=10)
        ax.set_xlabel('Delta Macro F1')
        ax.set_title('Feature Group Importance — Drop-One-Group (Tuned XGBoost, 3-class)')
        plt.tight_layout()
        plt.savefig('results/final/feature_ablation.png', bbox_inches='tight')
        plt.show()

        print("Feature group ranking (by impact when dropped):")
        for _, row in drops.iterrows():
            impact = "CRITICAL" if row['delta'] < -0.02 else "moderate" if row['delta'] < -0.005 else "marginal" if row['delta'] < 0 else "redundant"
            print(f"  {row['group']:15s}: delta={row['delta']:+.4f} ({impact})")
    else:
        print("No ablation results found in ml_tuned.csv")
except FileNotFoundError:
    print("ml_tuned.csv not found — run nb03c first.")


## 9. Summary Table for PFA Report

A clean, presentation-ready comparison table.


In [ ]:
# ── Build the final summary table ──
summary_rows = []

# Best ML per strategy
for strat in ['3class', 'binary']:
    ml_sub = master[(master['type'] == 'ML') & (master['strategy'] == strat)]
    if len(ml_sub) > 0:
        best = ml_sub.iloc[0]
        summary_rows.append({
            'Strategy': '3-class' if strat == '3class' else 'Binary',
            'Type': 'ML',
            'Best Model': best['model'],
            'CV Macro F1': f"{best['cv_f1_mean']:.4f} ± {best['cv_f1_std']:.4f}",
            'Phase': best['source'],
        })

    dl_sub = master[(master['type'] == 'DL') & (master['strategy'] == strat)]
    if len(dl_sub) > 0:
        best = dl_sub.iloc[0]
        summary_rows.append({
            'Strategy': '3-class' if strat == '3class' else 'Binary',
            'Type': 'DL',
            'Best Model': best['model'],
            'CV Macro F1': f"{best['cv_f1_mean']:.4f} ± {best['cv_f1_std']:.4f}",
            'Phase': best['source'],
        })

summary = pd.DataFrame(summary_rows)
print("SUMMARY TABLE — Best Models")
print("=" * 80)
print(summary.to_string(index=False))

# Add test results
print("\n\nFINAL TEST RESULTS")
print("=" * 80)
print(all_test.to_string(index=False))

# Save
summary.to_csv('results/final/summary_table.csv', index=False)
all_test.to_csv('results/final/test_results.csv', index=False)


## 10. Conclusions

### Finding 1: Problem framing >> Model selection >> Hyperparameter tuning

The single most impactful decision was **dropping the "operating" class**. Going from 4-class (F1 ≈ 0.44) to 3-class (F1 ≈ 0.67) or binary (F1 ≈ 0.75) was a gain of +0.23 to +0.31. By comparison, switching from RF to XGBoost on the same data gained +0.02, and Optuna tuning gained another +0.01.

"Operating" is not a real outcome — it means "no outcome yet." It overlaps with "closed" in feature space, making it nearly indistinguishable. SMOTE makes this worse by generating synthetic samples in the overlap zone.

### Finding 2: ML outperforms DL on this tabular dataset

This confirms the findings of Grinsztajn et al. (2022). On identical data with identical evaluation:
- **3-class:** ML best ≈ 0.667, DL best ≈ 0.63-0.66
- **Binary:** ML best ≈ 0.755, DL best ≈ 0.72-0.75

The gap is consistent but modest (2-5%). It's smaller on binary (simpler decision boundary) than on 3-class.

### Finding 3: Timing features matter most, funding amounts don't

Feature ablation revealed that `funding_recency_years`, `funding_duration_days`, and `company_age_years` carry the strongest signal. Dropping the timing group costs -0.045 F1. Dropping raw funding features actually *improves* F1 by +0.001 — once you know the timing and sector, the dollar amount is redundant.

### Finding 4: Ensembling provides modest but consistent gains

Stacking (XGBoost + LightGBM + RF → LogReg) achieved the best overall ML result. The gain over the best individual model is small (+0.003-0.005) because all base models are tree-based. Mixing in a DL model could provide more diversity.

### Finding 5: TabNet is the strongest DL architecture for tabular data

Among the 4 DL architectures tested (MLP, TabNet, TabTransformer, FT-Transformer), TabNet consistently performed best. Its attention-based feature selection is the closest neural mechanism to the feature splitting that makes trees effective.

### Recommendations for practitioners

1. Start by questioning your class definitions, not your model choice.
2. On tabular data with < 50K rows, try gradient boosting (XGBoost/LightGBM) first.
3. If DL is required, use TabNet — not a generic MLP.
4. Feature engineering matters: timing and sector signals beat raw financial metrics.


In [ ]:
# ── Save everything ──
print("All results saved to results/final/:")
for f in sorted(os.listdir('results/final')):
    size = os.path.getsize(f'results/final/{f}') / 1024
    print(f"  {f:45s} {size:6.1f} KB")

print("\n✓ Notebook 05 complete.")
print("  All charts saved to results/final/ for the PFA report.")
